<a href="https://colab.research.google.com/github/iDurugkar/practice-2026/blob/main/TorchCode/34_speculative_decoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

Implement the **acceptance/rejection step** of speculative decoding — a technique for accelerating LLM inference.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [2]:
import torch

In [16]:
# ✏️ YOUR IMPLEMENTATION HERE

def speculative_decode(target_probs, draft_probs, draft_tokens):
  # accept/reject loop
  accepted = []
  tps = torch.gather(target_probs, dim=1, index=draft_tokens.unsqueeze(1))
  dps = torch.gather(draft_probs, dim=1, index=draft_tokens.unsqueeze(1))
  ratios = tps / dps
  probs = torch.clamp(ratios, max=1.)
  mask = torch.bernoulli(probs).to(torch.bool)
  for i in range(len(mask)):
    if mask[i]:
      accepted.append(draft_tokens[i])
    else:
      accepted.append(torch.multinomial(torch.clamp(target_probs[i] - draft_probs[i], min=0.), 1))
      break
  return accepted


In [17]:
# 🧪 Debug
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('Random draft:', speculative_decode(target, draft, tokens))

Perfect draft: [tensor(2), tensor(5), tensor(1), tensor(8)]
Random draft: [tensor(2), tensor(5), tensor(1), tensor(8)]


In [18]:
# ✅ SUBMIT
from torch_judge import check
check('speculative_decoding')


🧪 Testing: Speculative Decoding (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Perfect draft: all accepted (10.0ms)
  ✅ [2/3] Output length bounded (42.9ms)
  ✅ [3/3] All tokens valid (16.3ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (69.1ms total)
  Progress saved. Run status() to see your dashboard.

